In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json, glob, pandas as pd, numpy as np, os
from pathlib import Path
from datetime import datetime

def ultimo_json(carpeta_base: str) -> str:
    """Devuelve el JSON más reciente por fecha real DD_MM_YYYY.json"""
    archivos = glob.glob(f'{carpeta_base}/**/*.json', recursive=True)
    assert len(archivos) > 0, f"❌ No hay JSONs en {carpeta_base}"
    def parse_fecha(ruta):
        try:
            return datetime.strptime(Path(ruta).stem, "%d_%m_%Y")
        except ValueError:
            return datetime.min
    return max(archivos, key=parse_fecha)

# ── ambiental ──
f = ultimo_json('../../data/ambiental')
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')


Archivo cargado: ../../data/ambiental/2026/04/14_04_2026.json
Keys disponibles: ['timestamp', 'observacion_meteorologica', 'calidad_aire']


In [2]:
#============================================
# Celda 2 — Explorar estructura AEMET
#============================================
aemet_raw  = data['observacion_meteorologica']
estaciones = aemet_raw['estaciones']

print(f'Total estaciones: {aemet_raw["total_estaciones"]}')
print(f'Timestamp captura: {aemet_raw["timestamp_captura"]}')
print(f'\nKeys de una estación:')
print(list(estaciones[0].keys()))
print(f'\nEjemplo estación[0]:')
print(json.dumps(estaciones[0], indent=2, ensure_ascii=False))


Total estaciones: 10136
Timestamp captura: 2026-04-14T09:43:21.915431

Keys de una estación:
['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr']

Ejemplo estación[0]:
{
  "idema": "0009X",
  "ubi": "ALFORJA",
  "lat": 41.213892,
  "lon": 0.963335,
  "alt": 406.0,
  "fint": "2026-04-13T21:00:00+0000",
  "ta": 10.5,
  "tamin": 10.5,
  "tamax": 11.4,
  "prec": 0.0,
  "hr": 54.0,
  "vv": 5.1,
  "vmax": 9.8,
  "dv": 281.0,
  "dmax": 270.0,
  "pres": null,
  "pres_nmar": null,
  "vis": null,
  "inso": null,
  "ts": null,
  "tpr": null
}


In [3]:
#============================================
# Celda 3 — Parsear AEMET a DataFrame
#============================================
df_aemet = pd.DataFrame(estaciones)

# Convertir timestamp
df_aemet['fint'] = pd.to_datetime(df_aemet['fint'], utc=True, errors='coerce')
df_aemet['fecha'] = df_aemet['fint'].dt.date

# Columnas numéricas
cols_num = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
            'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
for c in cols_num:
    if c in df_aemet.columns:
        df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

print(f'Shape: {df_aemet.shape}')
print(f'Columnas: {list(df_aemet.columns)}')
df_aemet.head()


Shape: (10136, 22)
Columnas: ['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr', 'fecha']


,idema,ubi,lat,lon,alt,fint,ta,tamin,tamax,prec,...,vmax,dv,dmax,pres,pres_nmar,vis,inso,ts,tpr,fecha
0,0009X,ALFORJA,41.213892,0.963335,406.0,2026-04-13 21:00:00+00:00,10.5,10.5,11.4,0.0,...,9.8,281.0,270.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-13
1,0016A,REUS AEROPUERTO,41.145000,1.163611,71.0,2026-04-13 21:00:00+00:00,14.8,14.8,15.7,0.0,...,14.9,310.0,270.0,1004.0,1013.1,30.0,0.0,13.7,2.3,2026-04-13
2,0034X,VALLS,41.293053,1.260838,233.0,2026-04-13 21:00:00+00:00,12.9,12.9,13.9,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-13
3,0042Y,TARRAGONA FAC. GEOGRAFIA,41.123892,1.249167,55.0,2026-04-13 21:00:00+00:00,14.1,14.1,15.2,0.0,...,6.9,297.0,340.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-13
4,0061X,PONTONS,41.417052,1.519269,632.0,2026-04-13 21:00:00+00:00,8.9,8.9,10.2,0.0,...,6.3,321.0,330.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-13


In [4]:
#============================================
# Celda 4 — Análisis rápido AEMET
#============================================
print('Valores nulos por columna:')
nulos = df_aemet.isnull().sum()
print(nulos[nulos > 0])

print(f'\nRango temperatura:  {df_aemet["ta"].min():.1f}°C — {df_aemet["ta"].max():.1f}°C')
print(f'Media temperatura:  {df_aemet["ta"].mean():.1f}°C')
print(f'\nEstaciones con precipitación > 0: {(df_aemet["prec"] > 0).sum()}')
print(f'Precipitación máxima: {df_aemet["prec"].max():.1f} mm')
print(f'\nRango altitud: {df_aemet["alt"].min():.0f}m — {df_aemet["alt"].max():.0f}m')
print(f'\nEstaciones sin coordenadas: {df_aemet[["lat","lon"]].isnull().any(axis=1).sum()}')


Valores nulos por columna:
ta            138
tamin         138
tamax         138
prec          259
hr            139
vv           1376
vmax         1413
dv           1337
dmax         1386
pres         7228
pres_nmar    8291
vis          8879
inso         8414
ts           8492
tpr          7177
dtype: int64

Rango temperatura:  -9.4°C — 22.6°C
Media temperatura:  8.8°C

Estaciones con precipitación > 0: 318
Precipitación máxima: 5.2 mm

Rango altitud: 0m — 2856m

Estaciones sin coordenadas: 0


In [5]:
#============================================
# Celda 5 — Explorar estructura WAQI
#============================================
waqi_raw = data['calidad_aire']
ciudades = waqi_raw['ciudades']

print(f'Total ciudades OK:  {waqi_raw["total_ciudades_ok"]}')
print(f'Total errores:      {waqi_raw["total_errores"]}')
print(f'\nKeys de una ciudad:')
print(list(ciudades[0].keys()))
print(f'\nEjemplo ciudad[0]:')
print(json.dumps(ciudades[0], indent=2, ensure_ascii=False))


Total ciudades OK:  30
Total errores:      3

Keys de una ciudad:
['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']

Ejemplo ciudad[0]:
{
  "ciudad": "madrid",
  "nombre": "Madrid",
  "lat": 40.4167754,
  "lon": -3.7037902,
  "timestamp": "2026-02-09 14:00:00",
  "aqi": 28,
  "dominante": "o3",
  "NO2": 10.1,
  "PM25": 21,
  "PM10": 13,
  "O3": 28.1,
  "SO2": 0.6,
  "CO": 0.1,
  "temperatura": 10.5,
  "humedad": 99,
  "presion": 1009,
  "viento": 10
}


In [6]:
#============================================
# Celda 6 — Parsear WAQI a DataFrame
#============================================
df_waqi = pd.DataFrame(ciudades)

# Convertir tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat','lon','aqi','NO2','PM25','PM10','O3','SO2','CO',
            'temperatura','humedad','presion','viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

print(f'Shape: {df_waqi.shape}')
print(f'Columnas: {list(df_waqi.columns)}')
df_waqi.head(10)


Shape: (30, 17)
Columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento
0,madrid,Madrid,40.416775,-3.703790,2026-02-09 14:00:00,28,o3,10.1,21.0,13.0,28.1,0.6,0.1,10.5,99.0,1009.0,10.0
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-04-14 11:00:00,16,o3,9.6,21.0,13.0,16.3,2.1,0.1,15.5,58.0,1018.8,0.5
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-02-13 02:00:00,34,pm25,1.9,34.0,12.0,29.3,1.6,0.1,13.8,63.0,NaN,0.6
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-04-14 10:00:00,43,pm25,11.5,43.0,13.0,20.2,1.6,0.1,14.5,70.0,1022.9,2.0
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-04-14 09:00:00,21,pm25,17.4,21.0,11.0,NaN,3.6,0.1,13.0,82.0,1021.0,3.0
5,zaragoza,"El Picarral, Zaragoza, Spain",41.670278,-0.871111,2026-04-14 10:00:00,27,o3,4.1,26.0,NaN,26.7,NaN,1.9,14.1,45.5,1019.2,15.0
6,malaga,"Carranque, Malaga, Spain",36.719640,-4.447500,2026-04-14 10:00:00,26,o3,9.2,26.0,9.0,26.4,2.1,0.1,15.8,59.2,1021.8,12.6
7,murcia,"San Basilio Murcia Ciudad, Murcia, Spain",37.993960,-1.144628,2026-04-14 11:00:00,26,o3,10.3,20.0,13.0,25.5,1.2,0.1,18.3,46.0,1018.7,8.0
8,palma,"foners, Baleares, Spain",39.571250,2.657028,2026-02-09 14:00:00,32,o3,5.1,30.0,15.0,31.7,1.9,0.1,16.1,56.0,1002.7,4.0
9,alicante,"Rabassa, Alacant, Valencia, Spain",38.351111,-0.513889,2026-02-13 04:00:00,29,o3,2.3,9.0,3.0,28.5,1.6,0.1,15.6,10.0,1021.7,2.4


In [7]:
#============================================
# Celda 7 — Análisis rápido WAQI
#============================================
def nivel_aqi(aqi):
    if pd.isna(aqi):       return 'Sin datos'
    elif aqi <= 50:        return '🟢 Bueno'
    elif aqi <= 100:       return '🟡 Moderado'
    elif aqi <= 150:       return '🟠 Insalubre sensibles'
    elif aqi <= 200:       return '🔴 Insalubre'
    else:                  return '🟣 Muy insalubre'

df_waqi['nivel_aqi'] = df_waqi['aqi'].apply(nivel_aqi)

print('Ciudades por nivel AQI:')
print(df_waqi['nivel_aqi'].value_counts().to_string())

print(f'\nTop 5 peor AQI:')
print(df_waqi.nlargest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nTop 5 mejor AQI:')
print(df_waqi.nsmallest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nValores nulos por columna:')
print(df_waqi.isnull().sum()[df_waqi.isnull().sum() > 0])


Ciudades por nivel AQI:
nivel_aqi
🟢 Bueno    30

Top 5 peor AQI:
                            nombre  aqi dominante  PM25  PM10   O3
                   Burgos I, Spain   45      pm10   9.0  45.0 16.3
           Ranilla, Sevilla, Spain   43      pm25  43.0  13.0 20.2
Lope de Vega, Vigo, Galicia, Spain   42      pm25  42.0  17.0 13.9
     Constitución, Asturias, Spain   38      pm25  38.0  27.0  3.7
                   Erie, Ohio, USA   38      pm25  38.0  14.0 26.4

Top 5 mejor AQI:
                                     nombre  aqi dominante  PM25  PM10    O3
Girona (Escola de Música), Catalunya, Spain    0        co  13.0   9.0 16.00
  Vila Velha - IBES, Espírito Santo, Brazil    2        o3   NaN   NaN  1.78
    Tarragona (Bonavista), Catalunya, Spain    8      pm10   9.0   8.0  5.70
     Barcelona (Eixample), Catalunya, Spain   16        o3  21.0  13.0 16.30
     Arco de ladrillo II, Valladolid, Spain   17      pm25  17.0   8.0 23.60

Valores nulos por columna:
NO2        2
PM25       3

In [8]:
#============================================
# Celda 8 — Resumen calidad de datos
#============================================
resumen = pd.DataFrame([
    {
        'tabla':     'aemet_estaciones',
        'filas':     len(df_aemet),
        'columnas':  len(df_aemet.columns),
        'periodos':  df_aemet['fint'].nunique(),
        'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
    },
    {
        'tabla':     'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique(),
        'nulos_%':   round(df_waqi.isnull().sum().sum()  / (df_waqi.shape[0]  * df_waqi.shape[1])  * 100, 1)
    },
])
print(resumen.to_string(index=False))


           tabla  filas  columnas  periodos  nulos_%
aemet_estaciones  10136        22        13     24.6
   waqi_ciudades     30        18        11      3.5


In [9]:
#============================================
# Celda 9 — Guardar parquets
#============================================
os.makedirs('../../output/ambiental/01-raw', exist_ok=True)

df_aemet.to_parquet('../../output/ambiental/01-raw/raw_aemet_estaciones.parquet', index=False)
print(f'✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet ({len(df_aemet)} filas)')

df_waqi.to_parquet('../../output/ambiental/raw_waqi_ciudades.parquet', index=False)
print(f'✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet ({len(df_waqi)} filas)')

print('\n🎉 Todos los parquets ambiental guardados en output/')


✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet (10136 filas)
✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet (30 filas)

🎉 Todos los parquets ambiental guardados en output/
